# YOLO-UDD v2.0 Training on Google Colab

Train YOLO-UDD v2.0 with free GPU acceleration!

**Setup Steps:**
1. Runtime → Change runtime type → GPU (T4)
2. Run cells sequentially
3. Monitor training progress

**Repository:** https://github.com/kshitijkhede/YOLO-UDD-v2.0

## 1️⃣ Clone Repository & Check GPU

In [3]:
# Clone the YOLO-UDD repository
!git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git
%cd YOLO-UDD-v2.0

# Verify GPU is available
!nvidia-smi

Cloning into 'YOLO-UDD-v2.0'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 73 (delta 22), reused 71 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 55.62 KiB | 1.36 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/YOLO-UDD-v2.0/YOLO-UDD-v2.0
/bin/bash: line 1: nvidia-smi: command not found


## 2️⃣ Install Dependencies

In [4]:
# Install PyTorch with CUDA support
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install project dependencies
!pip install -q -r requirements.txt

# Verify installation
import torch
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ CUDA version: {torch.version.cuda}")
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected! Enable GPU in Runtime → Change runtime type")

✓ PyTorch version: 2.8.0+cu126
✓ CUDA available: False
⚠️  No GPU detected! Enable GPU in Runtime → Change runtime type


## 3️⃣ Dataset Setup

**Choose one option:**

### Option A: Use Dummy Data (Quick Testing)
- Already configured
- Good for testing the pipeline
- No upload needed

### Option B: Upload Your Dataset
- Mount Google Drive
- Link to your TrashCAN dataset

In [ ]:
# Option A: Setup dummy data directories (auto-generates data)
import os
os.makedirs('data/trashcan/annotations', exist_ok=True)
os.makedirs('data/trashcan/images/train', exist_ok=True)
os.makedirs('data/trashcan/images/val', exist_ok=True)
print("✓ Dataset directories created")
print("✓ Will use dummy data for testing")

In [ ]:
# Option B: Mount Google Drive (if you have real dataset)
# Uncomment these lines if you want to use your own dataset

# from google.colab import drive
# drive.mount('/content/drive')
#
# # Link to your dataset in Google Drive
# # Adjust path to match your Drive structure
# !ln -s /content/drive/MyDrive/TrashCAN_Dataset data/trashcan

## 4️⃣ Test Model Architecture

In [ ]:
# Import and build the model
import sys
sys.path.append('/content/YOLO-UDD-v2.0')

from models import build_yolo_udd
import torch

# Build YOLO-UDD model
model = build_yolo_udd(num_classes=3, pretrained=None)
model_info = model.get_model_info()

print("="*60)
print("YOLO-UDD v2.0 Model Information")
print("="*60)
for key, value in model_info.items():
    print(f"{key}: {value}")
print("="*60)

# Test forward pass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
dummy_input = torch.randn(2, 3, 640, 640).to(device)

with torch.no_grad():
    predictions, turb_score = model(dummy_input)

print(f"\n✓ Model test successful!")
print(f"✓ Predictions: {len(predictions)} scales")
print(f"✓ Turbidity score shape: {turb_score.shape}")
print(f"✓ Device: {device}")

## 5️⃣ Configure Training

In [ ]:
# Training configuration
BATCH_SIZE = 16   # Increase if you have more GPU memory
EPOCHS = 50       # Adjust based on your needs
LEARNING_RATE = 0.01
NUM_WORKERS = 2   # Colab optimal value

print("🚀 Training Configuration:")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Device: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
print(f"   Workers: {NUM_WORKERS}")

## 6️⃣ Start Training 🎯

In [ ]:
# Run training with configured parameters
import os

# Use os.system with f-string to interpolate variables
cmd = f"python scripts/train.py --config configs/train_config.yaml --data-dir data/trashcan --batch-size {BATCH_SIZE} --epochs {EPOCHS} --lr {LEARNING_RATE} --save-dir runs/train"
print(f"🚀 Running: {cmd}")
os.system(cmd)

## 7️⃣ Monitor with TensorBoard 📊

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir runs/train/logs

## 8️⃣ View Training Results

In [ ]:
# List available checkpoints
import os
import glob

checkpoint_dir = 'runs/train/checkpoints'
if os.path.exists(checkpoint_dir):
    checkpoints = glob.glob(f"{checkpoint_dir}/*.pt")
    print("📦 Available checkpoints:")
    for ckpt in sorted(checkpoints):
        size_mb = os.path.getsize(ckpt) / (1024 * 1024)
        print(f"   {os.path.basename(ckpt)}: {size_mb:.2f} MB")
else:
    print("⏳ No checkpoints found yet. Training may still be in progress.")

## 9️⃣ Download Trained Model

In [ ]:
# Download best model
from google.colab import files

best_model = 'runs/train/checkpoints/best.pt'
if os.path.exists(best_model):
    print(f"📥 Downloading {best_model}...")
    files.download(best_model)
    print("✓ Download complete!")
else:
    print("❌ Best model not found. Check if training completed successfully.")

## 🔟 Save to Google Drive (Optional)

In [ ]:
# Copy all results to Google Drive for persistence
!mkdir -p /content/drive/MyDrive/YOLO-UDD-Results
!cp -r runs/train/* /content/drive/MyDrive/YOLO-UDD-Results/

print("✓ Results saved to Google Drive:")
print("  📁 MyDrive/YOLO-UDD-Results/")

---

## 🎉 Training Complete!

**Next Steps:**
1. Download your trained model (`best.pt`)
2. Run inference on new images
3. Evaluate on test set
4. Deploy for real-world use

**Questions?** Check the [GitHub repo](https://github.com/kshitijkhede/YOLO-UDD-v2.0) or open an issue.